# Logistic Regression — IEEE-CIS Fraud Detection

Metric: **ROC-AUC** and **PR-AUC** (fraud is ~3.5%, so accuracy is misleading).

## 1. Setup and authentication

In [1]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

import os

# --- Project ID is never hardcoded; it is read from Colab Secrets or a .env file ---
# Colab: open the key icon (Secrets) in the left sidebar > add GCP_PROJECT_ID.
# Local: put a .env in the project root (see .env.example) -> GCP_PROJECT_ID=...
PROJECT = None
try:
    from google.colab import userdata            # Colab Secrets
    PROJECT = userdata.get("GCP_PROJECT_ID")
except Exception:
    pass
if not PROJECT:
    try:
        from dotenv import load_dotenv            # local .env
        load_dotenv()
    except Exception:
        pass
    PROJECT = os.environ.get("GCP_PROJECT_ID")
if not PROJECT:
    raise ValueError(
        "GCP_PROJECT_ID not found. Set it via Colab Secrets (key icon) or, "
        "locally, in a .env file."
    )

RANDOM_STATE = 42
client = bigquery.Client(project=PROJECT)
print("Ready.")

Ready.


## 2. Load data + memory optimization

In [2]:
df = client.query(f"SELECT * FROM `{PROJECT}.fraud.train_zengin`").to_dataframe()

# For memory: float64 -> float32, int64 -> int32
for c in df.columns:
    if df[c].dtype == 'float64':
        df[c] = df[c].astype('float32')
    elif df[c].dtype == 'int64':
        df[c] = df[c].astype('int32')

print("Data loaded. Shape:", df.shape)

Data loaded. Shape: (590540, 409)


### 3. Data preparation

Numeric columns: impute missing + scale. Categorical (String) columns: One-Hot Encode (`card4`, `card6`, `P_emaildomain`, `DeviceType`, `DeviceInfo`, `M1–M9`, `id_*` ...). All transforms are **fit on the training set only, after the split** → no data leakage. Numeric columns are `float32` to save RAM.

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
import gc

y = df['isFraud']
X = df.drop(columns=['isFraud', 'TransactionID', 'uid'], errors='ignore')
del df; gc.collect()

numeric_features = X.select_dtypes(include=['int32', 'float32']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

# Prevent the int32/float32 mix from upcasting to float64 (saves RAM)
X[numeric_features] = X[numeric_features].astype('float32')

# Split first, then fit -> no data leakage
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
del X; gc.collect()

# Numeric: fill missing with 0 + scale (fit on training set only)
imputer = SimpleImputer(strategy='constant', fill_value=0)
scaler = StandardScaler()
train_num = scaler.fit_transform(imputer.fit_transform(X_train[numeric_features]))
val_num = scaler.transform(imputer.transform(X_val[numeric_features]))

# Categorical: fill missing + One-Hot (min_frequency groups rare values into one bucket)
cat_imputer = SimpleImputer(strategy='constant', fill_value='missing')
ohe = OneHotEncoder(handle_unknown='ignore', min_frequency=0.001,
                    sparse_output=False, dtype=np.float32)
train_cat = ohe.fit_transform(cat_imputer.fit_transform(X_train[categorical_features]))
val_cat = ohe.transform(cat_imputer.transform(X_val[categorical_features]))

# Combine numeric + categorical (all float32)
X_train_final = np.hstack([train_num, train_cat])
X_val_final = np.hstack([val_num, val_cat])
del train_num, val_num, train_cat, val_cat, X_train, X_val; gc.collect()

print("Ready ->", X_train_final.shape, X_val_final.shape)

Ready -> (472432, 829) (118108, 829)


### 4. Logistic Regression training and evaluation

In [4]:
# class_weight='balanced' -> for imbalanced data
lr_model = LogisticRegression(max_iter=1000, class_weight='balanced',
                              random_state=RANDOM_STATE)
lr_model.fit(X_train_final, y_train)

lr_proba = lr_model.predict_proba(X_val_final)[:, 1]
lr_roc = roc_auc_score(y_val, lr_proba)
lr_pr = average_precision_score(y_val, lr_proba)

print("=== Logistic Regression Results ===")
print(f"  ROC-AUC : {lr_roc:.4f}")
print(f"  PR-AUC  : {lr_pr:.4f}")

=== Logistic Regression Results ===
  ROC-AUC : 0.8847
  PR-AUC  : 0.4648


## Summary

The scores are in the output of the cell above. This version removes data leakage (split first, then fit on the training set only) and adds String columns via One-Hot Encoding.

Since logistic regression is a linear model, it serves as a solid **baseline** for tree-based LightGBM/XGBoost. Possible improvements: `C` search, `OneHotEncoder` cardinality tuning, or target encoding on high-cardinality columns.